# 📦 Cenário de Dados: Sistema de Despachos

Este cenário simula o banco de dados de um aplicativo de logística e despachos. Os dados são altamente transacionais (mudanças constantes de status de motoristas e corridas), justificando o uso de tecnologias Lakehouse como o Delta Lake para garantir transações ACID.

## Modelo Entidade-Relacionamento (ER)
Abaixo, o relacionamento entre os motoristas cadastrados e as corridas realizadas.

```mermaid
erDiagram
    MOTORISTAS ||--o{ CORRIDAS : "realiza"
    
    MOTORISTAS {
        int id_motorista PK
        string nome
        string veiculo
        string status
    }
    
    CORRIDAS {
        int id_corrida PK
        int id_motorista FK
        string origem
        string destino
        float valor_corrida
        string status_corrida
    }

In [ ]:
import os
import urllib.request
import pyspark
from delta import *

# =====================================================================
# 🛠️ CONSERTO PARA WINDOWS: Baixando o "tradutor" do Spark (Hadoop)
# =====================================================================
hadoop_home = os.path.abspath("hadoop_win")
hadoop_bin = os.path.join(hadoop_home, "bin")
os.makedirs(hadoop_bin, exist_ok=True)

base_url = "https://raw.githubusercontent.com/cdarlint/winutils/master/hadoop-3.2.0/bin/"

# Baixa os arquivos winutils.exe e hadoop.dll se você ainda não tiver
for file in ["winutils.exe", "hadoop.dll"]:
    file_path = os.path.join(hadoop_bin, file)
    if not os.path.exists(file_path):
        print(f"Baixando {file} para o Windows parar de reclamar...")
        urllib.request.urlretrieve(base_url + file, file_path)

# Avisa o Windows e o Spark onde estão os arquivos
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] += os.pathsep + hadoop_bin
# =====================================================================


# 1. Configurando o Spark para usar o Delta Lake
print("Ligando o motor do Apache Spark...")
builder = pyspark.sql.SparkSession.builder.appName("DeltaLake_Despachos") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0")

spark = configure_spark_with_delta_pip(builder).getOrCreate()
spark.sparkContext.setLogLevel("ERROR") # Esconde os avisos vermelhos gigantes

print("✅ Spark com Delta Lake iniciado com sucesso!")

# 2. Comandos DDL: Criando as Tabelas
spark.sql("""
CREATE TABLE IF NOT EXISTS motoristas (
    id_motorista INT,
    nome STRING,
    veiculo STRING,
    status STRING
) USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS corridas (
    id_corrida INT,
    id_motorista INT,
    origem STRING,
    destino STRING,
    valor_corrida FLOAT,
    status_corrida STRING
) USING DELTA
""")

print("✅ Tabelas 'motoristas' e 'corridas' criadas no formato Delta!")

Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.RuntimeException: java.io.FileNotFoundException: java.io.FileNotFoundException: HADOOP_HOME and hadoop.home.dir are unset. -see https://wiki.apache.org/hadoop/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:735)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:270)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1139)
	at org.apache.hadoop.fs.FileUtil.chmod(FileUtil.java:1125)
	at org.apache.spark.util.Utils$.fetchFile(Utils.scala:489)
	at org.apache.spark.SparkContext.addFile(SparkContext.scala:1795)
	at org.apache.spark.SparkContext.$anonfun$new$16(SparkContext.scala:533)
	at org.apache.spark.SparkContext.$anonfun$new$16$adapted(SparkContext.scala:533)
	at scala.collection.immutable.List.foreach(List.scala:431)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:533)
	at org.apache.spark.api.java.JavaSparkC